# Leson 11 - Model Context Protocol (MCP)

The **Model Context Protocol (MCP)** na open standard wey dey allow agents to find and use tools, resources, and data sources dynamically at runtime. Instead of hardcoding tools into an agent, MCP dey let agents connect to external servers wey dey expose capabilities on demand.

In this leson, you go learn:
- Wetin MCP be and why e matter for agent systems
- How the MCP client-server architecture dey work
- How to build agents wey dey use MCP-style tool discovery


## Setup

**Wetin you need:**
- Azure AI Foundry project wey don get deployed model
- Run `az login` make you sign in


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Wetin be di Model Context Protocol (MCP)?

MCP dey define one standard way for AI agents to find and interact wit external tools and data sources:

- **MCP Server**: E dey expose tools, resources, and prompts through a standard protocol
- **MCP Client**: Na di agent runtime wey dey connect to servers and dey find out di available capabilities
- **Dynamic Discovery**: Agents no need hardcoded tools — dem go discover wetin dey available at runtime

Dis powerful for building extensible agent systems wey new capabilities fit add without changing di agent code.


## How MCP Dey Work

```
┌─────────────┐     discover      ┌─────────────────┐
│  MCP Client  │ ──────────────► │   MCP Server     │
│  (Agent)     │                  │  (Tool Provider) │
│              │ ◄────────────── │                   │
│              │   tool results   │  • list_tools()  │
│              │                  │  • call_tool()   │
└─────────────┘                  │  • resources     │
                                  └─────────────────┘
```

1. The agent (MCP client) dey connect to an MCP server
2. The server dey respond with a list of available tools and their schemas
3. The agent fit call any discovered tool as e dey reason
4. Results dey flow back through the same protocol


## Dey simulate MCP Tool Discovery

Because real MCP server need a running server process, we go show the pattern by using `@tool` functions wey dey simulate wetin an MCP-connected accommodation service for provide.

For production, these tools go dey discovered dynamically from an MCP server instead of being defined locally.


In [ ]:
@tool(approval_mode="never_require")
def search_accommodations(
    location: Annotated[str, "The city to search for accommodations"],
    check_in: Annotated[str, "Check-in date (YYYY-MM-DD)"],
    check_out: Annotated[str, "Check-out date (YYYY-MM-DD)"],
    guests: Annotated[int, "Number of guests"] = 2
) -> str:
    """Search for accommodations (simulating an MCP-connected Airbnb tool).
    In production, this would be discovered via MCP from an accommodation service."""
    listings = {
        "Tokyo": [
            {"name": "Shinjuku Modern Apartment", "price": 120, "rating": 4.8},
            {"name": "Traditional Ryokan in Asakusa", "price": 200, "rating": 4.9},
            {"name": "Shibuya Studio", "price": 85, "rating": 4.5},
        ],
        "Paris": [
            {"name": "Le Marais Charming Flat", "price": 150, "rating": 4.7},
            {"name": "Montmartre Artist Loft", "price": 110, "rating": 4.6},
        ],
        "Barcelona": [
            {"name": "Gothic Quarter Penthouse", "price": 130, "rating": 4.8},
            {"name": "Barceloneta Beach Flat", "price": 95, "rating": 4.4},
        ],
    }
    results = listings.get(location, [])
    if not results:
        return f"No accommodations found in {location}"
    output = f"Accommodations in {location} ({check_in} to {check_out}, {guests} guests):\n"
    for listing in results:
        output += f"  - {listing['name']}: ${listing['price']}/night (★{listing['rating']})\n"
    return output


@tool(approval_mode="never_require")
def get_local_experiences(
    location: Annotated[str, "The city to find experiences in"],
    interest: Annotated[str, "Type of experience (food, culture, adventure, etc.)"] = "all"
) -> str:
    """Get local experiences and activities (simulating an MCP-connected tourism tool)."""
    experiences = {
        "Tokyo": {
            "food": ["Tsukiji Market Tour ($45)", "Ramen Making Class ($60)", "Sake Tasting ($35)"],
            "culture": ["Tea Ceremony ($50)", "Samurai Museum ($15)", "Sumo Tournament ($80)"],
            "adventure": ["Mt. Fuji Day Trip ($120)", "Go-kart City Tour ($80)"],
        },
        "Paris": {
            "food": ["Wine & Cheese Tasting ($55)", "Cooking Class ($90)", "Market Tour ($40)"],
            "culture": ["Louvre Guided Tour ($35)", "Montmartre Art Walk ($25)"],
        },
    }
    city_exp = experiences.get(location, {})
    if not city_exp:
        return f"No experiences found in {location}"
    if interest != "all" and interest in city_exp:
        items = city_exp[interest]
        return f"{interest.title()} experiences in {location}:\n" + "\n".join(f"  - {e}" for e in items)
    output = f"All experiences in {location}:\n"
    for cat, items in city_exp.items():
        output += f"\n  {cat.title()}:\n"
        for item in items:
            output += f"    - {item}\n"
    return output

## How to make an agent wit MCP-style tools


In [ ]:
agent = await provider.create_agent(
    tools=[search_accommodations, get_local_experiences],
    name="AccommodationAgent",
    instructions="""You are an accommodation and travel experiences specialist powered by MCP-connected services.

Help travelers find the perfect place to stay and things to do. When searching:
1. Use the search_accommodations tool to find listings
2. Use the get_local_experiences tool to suggest activities
3. Compare options and make personalized recommendations
4. Consider the traveler's budget, interests, and travel style""",
)

response = await agent.run(
    "I'm visiting Tokyo for 5 nights in April with my partner. We love traditional Japanese culture and food. "
    "Find us a place to stay and suggest some experiences.",
    )
print(response)

## MCP for Production

For production environment, MCP dey enable powerful patterns:

- **Dynamic tool discovery**: Agents dey connect to MCP servers an find tools as dem dey run
- **Decoupled architecture**: Tool providers fit update on dia own without depend on the agent
- **Cross-organization sharing**: Teams fit expose wetin dem sabi do via MCP servers we any agent fit use
- **Microsoft Agent Framework support**: MAF get built-in MCP client support via the `mcp` integration

To use a real MCP server with MAF, you would connect via `hosted_mcp_tool()` or the MCP client integration.

**Find out more:**
- [MCP Spesifikeshon](https://modelcontextprotocol.io/)
- [Microsoft Agent Framework MCP Support](https://github.com/microsoft/agent-framework/tree/main/python/samples/02-agents/mcp)


## Summary

For this lesson, you don learn:
- **MCP** na open standard wey dey allow agents and tool providers find tools dynamically
- The **client-server architecture** dey let agents find wetin dem fit do at runtime
- MCP dey enable **extensible, decoupled agent systems** wey tools fit add without changing code
- Microsoft Agent Framework get **built-in MCP support** for production use


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
Disclaimer:
Dis dokument na AI wey translate am wit di service [Co-op Translator] (https://github.com/Azure/co-op-translator). Even tho we dey try make am correct, abeg note say automated translations fit get errors or inaccuracies. Di original dokument for im own language suppose be di main authority. For critical mata, better make person wey be professional human translator do am. We no dey liable for any misunderstanding or misinterpretation wey fit come from using dis translation.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
